In [ ]:
import rclpy as rp
from turtlesim.msg import Pose

rp.init()

test_node = rp.create_node('sub_test')

cnt = 0
def callback(data):
    global cnt
    cnt += 1
    print("X: ", data.x)
    print("Theta", data.theta)
    
    if data.x >= 7:
        print("오른쪽 끝으로 이동 중")

    if cnt == 5:
        test_node.destroy_node()
        rp.shutdown()
    

test_node.create_subscription(Pose, "/turtle1/pose", callback, 10) # 메세지 타입, 토픽이름, call back 함수, 큐 크기 

rp.spin(test_node) # 이벤트가 발생하는지 내부적으로 계속 확인 

In [ ]:
import rclpy as rp
from geometry_msgs.msg import Twist

rp.init()

test_node = rp.create_node('pub_test')

msg = Twist()

msg.linear.x = 3.0
msg.angular.z = 1.57

# "/turtle1/cmd_vel Topic에 데이터를 보내는 Publisher 생성"
# "이 Topic에 데이터를 계속 보낼게"
pub = test_node.create_publisher(Twist, '/turtle1/cmd_vel', 10)


cnt = 0
def timer_callback():
    global cnt
    cnt += 1

    print(cnt)
    pub.publish(msg) # 명령을 Topic 채널에 보내는 것이다.

    if cnt >= 10:
        test_node.destroy_node()
        rp.shutdown()
        

timer_period = 0.5
timer = test_node.create_timer(timer_period, timer_callback)
rp.spin(test_node)




In [1]:
import rclpy as rp
from turtlesim.srv import TeleportAbsolute

rp.init()

test_node = rp.create_node('client_test')

service_name = '/turtle1/teleport_absolute'

# 이제 이 객체(cli)로 서비스 요청을 보낼 수 있다
cli = test_node.create_client(
    TeleportAbsolute,
    service_name
)

while not cli.wait_for_service(timeout_sec = 1.0):
    print("서비스 기다리는 중...")

# 서비스에 보낼 요청(Request) 객체 생성.
req1 = TeleportAbsolute.Request()

req1.x = 2.0
req1.y = 2.0
req1.theta = 0.0

# 요청 보내고, 바로 다음 코드 실행 가능
# 나중에 결과가 들어올 예정인 객체
future1 = cli.call_async(req1)

rp.spin_until_future_complete(test_node, future1)

req2 = TeleportAbsolute.Request()

req2.x = 8.0
req2.y = 8.0
req2.theta = 1.57

future2 = cli.call_async(req2)

# 서비스 응답이 올 때까지 Node를 계속 돌린다.
rp.spin_until_future_complete(test_node, future2)

# 종료
test_node.destroy_node()
rp.shutdown()

2026-05-13 17:29:47.113 [RTPS_TRANSPORT_SHM Error] Failed init_port fastrtps_port11161: open_and_lock_file failed -> Function open_port_internal
서비스 기다리는 중...
서비스 기다리는 중...
서비스 기다리는 중...
서비스 기다리는 중...


In [1]:
import numpy as np

class TurtleRobot:
    def __init__(self, name, x, y, theta, speed):
        self.name = name
        self.x = x
        self.y = y
        self.theta = theta
        self.speed = speed

    def move(self, duration):
        distance = self.speed * duration

        self.x += np.cos(self.theta) * distance
        self.y += np.sin(self.theta) * distance

    def rotate(self, angle):
        self.theta += angle

    def show_status(self):
        print("로봇 이름", self.name)
        print("현재 위치", self.x, self.y)
        print("현재 방향 출력", self.theta)
        print("현재 속도 출력", self.speed)


robot = TurtleRobot("turtle1", 0, 0, 0, 1.0)
# 객체 내부 상태 출력 
print(robot.__dict__)

{'name': 'turtle1', 'x': 0, 'y': 0, 'theta': 0, 'speed': 1.0}


In [5]:
# 객체 생성 후 속성값을 직접 변경
robot.speed = 5
print(robot.__dict__)

# 속도 변경 후 다시 move() 를 실행해본다.
robot.move(1)
print(robot.__dict__)

{'name': 'turtle1', 'x': np.float64(5.0), 'y': np.float64(0.0), 'theta': 0, 'speed': 5}
{'name': 'turtle1', 'x': np.float64(10.0), 'y': np.float64(0.0), 'theta': 0, 'speed': 5}


In [6]:
# CameraRobot 클래스를 생성한다.
class CameraRobot(TurtleRobot):
    def __init__(self, name, x, y, theta, speed, resolution):
        super().__init__(name, x, y, theta, speed)
        self.resolution = resolution
    
    def capture(self): 
        print("camera capture complete")

    def show_status(self):
        super().show_status()
        print("Camera Resolution:", self.resolution)


cam = CameraRobot("camrobot1", 0, 0, 0, 1.0, "1920 x 1080")
print(cam.__dict__)

{'name': 'camrobot1', 'x': 0, 'y': 0, 'theta': 0, 'speed': 1.0, 'resolution': '1920 x 1080'}


In [7]:
cam.show_status()

로봇 이름 camrobot1
현재 위치 0 0
현재 방향 출력 0
현재 속도 출력 1.0
Camera Resolution: 1920 x 1080
